# Exportar `retrain.pipeline` con Elyra (CLI)

Este notebook:

1. Localiza `retrain.pipeline` (en `workbench/pipeline/` o junto a este notebook).
2. Lista las **configuraciones de runtime** de Elyra y elige una compatible con **Kubeflow Pipelines** (la que usa tu `.pipeline`).
3. Ejecuta **`elyra-pipeline export`** y deja el YAML (y posiblemente artefactos COS) según tu runtime.

**Nota:** En OpenShift AI, `odh-elyra` suele generar YAML orientado a **Data Science Pipelines (KFP v2)**, no al recurso **Tekton** `Pipeline` (`tekton.dev/v1`) del taller. Para Tekton sigue usando `deployment/pipeline/pipeline.yaml` con `oc apply`.

Requisitos: imagen workbench con Elyra (`elyra-pipeline`, `elyra-metadata` en el PATH) y al menos un runtime creado (consola *Runtimes* o flujo de OpenShift AI).

**Validación de parámetros:** `elyra-pipeline export` exige valores en parámetros **required** del `.pipeline` (p. ej. `s3endpoint`). Este cuaderno rellena vacíos con el endpoint MinIO del taller y escribe un **staging** `.pipeline` en el mismo directorio que el original para no romper rutas a `step-*.ipynb`.

In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

# Directorio de salida del export (relativo al cwd de Jupyter, normalmente /opt/app-root/src)
OUTPUT_DIR = Path(os.environ.get("ELYRA_EXPORT_OUTPUT_DIR", "elyra_pipeline_export")).resolve()

# Si conoces el nombre exacto del runtime, asígnalo aquí; None = elegir automáticamente
RUNTIME_NAME_OVERRIDE: str | None = os.environ.get("ELYRA_RUNTIME_NAME") or None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("RUNTIME_NAME_OVERRIDE:", RUNTIME_NAME_OVERRIDE)

In [ ]:
import os


def find_retrain_pipeline() -> Path:
    """Busca retrain.pipeline en rutas habituales del workbench RHOAI."""
    candidates = [
        Path.cwd() / "workbench" / "pipeline" / "retrain.pipeline",
        Path.cwd() / "pipeline" / "retrain.pipeline",
        Path("/opt/app-root/src/workbench/pipeline/retrain.pipeline"),
        Path("/opt/app-root/src/pipeline/retrain.pipeline"),
    ]
    root = os.environ.get("JUPYTER_SERVER_ROOT")
    if root:
        r = Path(root)
        candidates.insert(0, r / "workbench" / "pipeline" / "retrain.pipeline")
        candidates.insert(1, r / "pipeline" / "retrain.pipeline")
    # Si el notebook vive en workbench/, el .pipeline está en pipeline/retrain.pipeline
    try:
        from IPython import get_ipython

        ip = get_ipython()
        if ip is not None:
            fn = ip.user_ns.get("__vsc_ipynb_file__")
            if isinstance(fn, str) and fn.endswith(".ipynb"):
                nb_dir = Path(fn).resolve().parent
                candidates.insert(0, nb_dir / "pipeline" / "retrain.pipeline")
                candidates.insert(1, nb_dir / "retrain.pipeline")
    except Exception:
        pass

    for p in candidates:
        if p is not None and p.is_file():
            return p.resolve()
    raise FileNotFoundError(
        "No se encontró retrain.pipeline. Coloca este notebook en workbench/ o define PIPELINE_PATH manualmente en la siguiente celda."
    )


PIPELINE_PATH = find_retrain_pipeline()
print("PIPELINE_PATH:", PIPELINE_PATH)

In [ ]:
# Si la celda anterior falló, descomenta y ajusta la ruta absoluta:
# PIPELINE_PATH = Path("/opt/app-root/src/workbench/pipeline/retrain.pipeline")

assert PIPELINE_PATH.is_file(), PIPELINE_PATH

In [ ]:
for exe in ("elyra-pipeline", "elyra-metadata"):
    path = shutil.which(exe)
    if not path:
        raise RuntimeError(
            f"No está en el PATH: {exe}. Esta imagen workbench debe incluir Elyra."
        )
    print(f"{exe}: {path}")

### Log del servidor: «No components could be found in any catalog for platform type 'KUBEFLOW_PIPELINES'»

Eso indica que **no hay ningún *Pipeline component catalog*** registrado para Kubeflow/Data Science Pipelines. Elyra intenta rellenar la caché de componentes y **no encuentra ninguno**; el export puede **quedarse mucho rato** o no terminar bien.

La línea de *airflow* (`No entrypoint with name 'airflow'`) es **normal** en `odh-elyra` si no instalaste el procesador Airflow: se puede ignorar.

**Log del contenedor (`oc logs`):** Elyra del **servidor Jupyter** puede registrar el mismo aviso al **arrancar** el workbench o al abrir el editor de pipelines, **antes** de que ejecutes la celda de catálogos. Si tras ejecutar esa celda ves **≥1 catálogo** y el export acaba sin error, esas entradas antiguas en el log suelen ser **ruido**; para que la UI deje de quejarse al inicio, el catálogo debe existir en metadatos **antes** del próximo arranque del servidor (o reinicia el workbench tras crear el catálogo).

**Qué hacer:** la celda siguiente **lista** los catálogos y, si no hay ninguno, intenta **`elyra-metadata create component-catalogs`** con un **catálogo URL mínimo** (un YAML de los ejemplos oficiales de Elyra en GitHub), igual que en la [guía CLI de componentes](https://elyra.readthedocs.io/en/stable/user_guide/pipeline-components.html). Desactiva eso con `ELYRA_SKIP_AUTO_CATALOG=1` si prefieres configurarlo solo en la UI (**Pipeline Components** en JupyterLab). Si el workbench **no puede salir a GitHub**, define `ELYRA_AUTO_CATALOG_URL` apuntando a un `.yaml` de componente KFP accesible desde el pod.

In [ ]:
import json

from elyra.metadata.manager import MetadataManager
from elyra.metadata.schemaspaces import ComponentCatalogs

# YAML público de la guía Elyra; sirve para que exista al menos un catálogo KFP.
_DEFAULT_KFP_CATALOG_URL = os.environ.get(
    "ELYRA_AUTO_CATALOG_URL",
    "https://raw.githubusercontent.com/elyra-ai/examples/main/component-catalog-connectors/kfp-example-components-connector/kfp_examples_connector/resources/filter_text_using_shell_and_grep.yaml",
)
_AUTO_CATALOG_NAME = os.environ.get("ELYRA_AUTO_CATALOG_NAME", "rhods_tl_kfp_minimal_url")

_cc_mm = MetadataManager(schemaspace=ComponentCatalogs.COMPONENT_CATALOGS_SCHEMASPACE_NAME)
_component_catalogs = _cc_mm.get_all(include_invalid=False)

print(f"Component catalogs configurados: {len(_component_catalogs)}")
for _cat in _component_catalogs:
    print(f"  - {_cat.name!r} (schema={_cat.schema_name!r})")

_skip = os.environ.get("ELYRA_SKIP_AUTO_CATALOG", "").strip().lower() in ("1", "true", "yes")
_named_exists = any(_c.name == _AUTO_CATALOG_NAME for _c in _component_catalogs)

if len(_component_catalogs) == 0 and not _skip:
    cmd = [
        "elyra-metadata",
        "create",
        "component-catalogs",
        "--name",
        _AUTO_CATALOG_NAME,
        "--display_name",
        "RHODS transfer-learning (catálogo URL mínimo)",
        "--runtime_type",
        "KUBEFLOW_PIPELINES",
        "--schema_name",
        "url-catalog",
        "--paths",
        json.dumps([_DEFAULT_KFP_CATALOG_URL]),
    ]
    print(
        "\nSin catálogos: creando uno mínimo (véase Elyra — Pipeline components / elyra-metadata create).\n"
        "Ejecutando:",
        subprocess.list2cmdline(cmd),
        "\n",
    )
    _proc = subprocess.run(cmd, capture_output=True, text=True)
    if _proc.stdout.strip():
        print(_proc.stdout.strip())
    if _proc.stderr.strip():
        print(_proc.stderr.strip())
    if _proc.returncode != 0:
        print(
            f"elyra-metadata create devolvió {_proc.returncode}. "
            "Si el clúster no tiene salida a GitHub, define ELYRA_AUTO_CATALOG_URL "
            "o añade un catálogo en Jupyter (Pipeline Components)."
        )
    _component_catalogs = _cc_mm.get_all(include_invalid=False)
    print(f"Tras intento automático: {len(_component_catalogs)} catálogo(s)")
    for _cat in _component_catalogs:
        print(f"  - {_cat.name!r} (schema={_cat.schema_name!r})")

if len(_component_catalogs) == 0:
    if _skip:
        print(
            "\n>>> ELYRA_SKIP_AUTO_CATALOG: no se creó catálogo automático. "
            "Sin catálogos, el export suele fallar con KUBEFLOW_PIPELINES en el log.\n"
        )
    else:
        print(
            "\n>>> Sin catálogos: el export suele fallar o tardar con el error "
            "KUBEFLOW_PIPELINES en el log. Configura catálogos en la UI o usa "
            "deployment/pipeline/pipeline.yaml para Tekton.\n"
        )


In [ ]:
from elyra.metadata.manager import MetadataManager
from elyra.metadata.schemaspaces import Runtimes

mm = MetadataManager(schemaspace=Runtimes.RUNTIMES_SCHEMASPACE_NAME)
all_runtimes = mm.get_all(include_invalid=False)

if not all_runtimes:
    raise RuntimeError(
        "No hay runtimes Elyra. Crea uno en Jupyter (Runtimes / metadatos) conectado a tu pipeline server y S3, "
        "como indica la documentación de OpenShift AI para pipelines en JupyterLab."
    )

rows = []
for m in all_runtimes:
    rt = (m.metadata or {}).get("runtime_type")
    rows.append(
        {
            "name": m.name,
            "display_name": m.display_name,
            "schema_name": m.schema_name,
            "runtime_type": rt,
        }
    )

try:
    from IPython.display import display

    import pandas as pd  # type: ignore

    display(pd.DataFrame(rows))
except Exception:
    for r in rows:
        print(r)

kfp_names = [
    r["name"]
    for r in rows
    if r["runtime_type"] == "KUBEFLOW_PIPELINES"
]
if not kfp_names:
    kfp_names = [r["name"] for r in rows]

chosen = RUNTIME_NAME_OVERRIDE if RUNTIME_NAME_OVERRIDE else kfp_names[0]
if RUNTIME_NAME_OVERRIDE and RUNTIME_NAME_OVERRIDE not in {r["name"] for r in rows}:
    raise ValueError(f"RUNTIME_NAME_OVERRIDE no existe: {RUNTIME_NAME_OVERRIDE!r}")

RUNTIME_NAME = chosen
print("Runtime seleccionado para export:", RUNTIME_NAME)

In [ ]:
import json

# Elyra valida parámetros obligatorios antes de exportar; s3endpoint no puede quedar vacío.
# El fichero temporal va en el MISMO directorio que retrain.pipeline: el CLI resuelve
# step-01.ipynb etc. respecto al directorio del .pipeline.

DEFAULT_PARAM_VALUES = {
    "s3endpoint": os.environ.get(
        "ELYRA_PARAM_S3ENDPOINT",
        "http://minio-service.central.svc:9000",
    ),
}


def pipeline_with_filled_required_params(src: Path) -> Path:
    data = json.loads(src.read_text(encoding="utf-8"))
    for pl in data.get("pipelines", []):
        props = pl.get("app_data", {}).get("properties", {})
        params = props.get("pipeline_parameters")
        if not isinstance(params, list):
            continue
        for param in params:
            name = param.get("name")
            if not name:
                continue
            dv = param.get("default_value")
            if not isinstance(dv, dict):
                dv = {"type": "String", "value": ""}
                param["default_value"] = dv
            val = dv.get("value")
            empty = val is None or (isinstance(val, str) and val.strip() == "")
            if not empty:
                continue
            if name in DEFAULT_PARAM_VALUES:
                dv["value"] = DEFAULT_PARAM_VALUES[name]

    staging = src.parent / ".retrain_export_staging.pipeline"
    staging.write_text(json.dumps(data, indent=2), encoding="utf-8")
    print("PIPELINE_EXPORT_PATH (staging):", staging)
    return staging


PIPELINE_EXPORT_PATH = pipeline_with_filled_required_params(PIPELINE_PATH)

In [ ]:
EXPORT_FORMAT = os.environ.get("ELYRA_EXPORT_FORMAT", "yaml")

cmd = [
    "elyra-pipeline",
    "export",
    str(PIPELINE_EXPORT_PATH),
    "--runtime-config",
    RUNTIME_NAME,
    "--format",
    EXPORT_FORMAT,
    "--output",
    str(OUTPUT_DIR),
    "--overwrite",
]
print("Ejecutando:", subprocess.list2cmdline(cmd))

proc = subprocess.run(cmd, capture_output=False, text=True)
if proc.returncode != 0:
    raise RuntimeError(f"elyra-pipeline export falló con código {proc.returncode}")

PIPELINE_EXPORT_PATH.unlink(missing_ok=True)

print("Listado en", OUTPUT_DIR, ":")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" ", p.name)

## Variables de entorno opcionales

| Variable | Efecto |
|----------|--------|
| `ELYRA_RUNTIME_NAME` | Fuerza el nombre del runtime (igual que `RUNTIME_NAME_OVERRIDE` en código). |
| `ELYRA_EXPORT_OUTPUT_DIR` | Directorio de salida (por defecto `elyra_pipeline_export` bajo el cwd). |
| `ELYRA_EXPORT_FORMAT` | `yaml` o `py` según lo que admita tu runtime (por defecto `yaml`). |
| `ELYRA_PARAM_S3ENDPOINT` | Valor por defecto para el parámetro obligatorio `s3endpoint` si el `.pipeline` lo tiene vacío (por defecto `http://minio-service.central.svc:9000`). |
| `ELYRA_SKIP_AUTO_CATALOG` | Si es `1`/`true`/`yes`, no ejecuta `elyra-metadata create` cuando no hay catálogos KFP. |
| `ELYRA_AUTO_CATALOG_URL` | URL del YAML de componente usado para el catálogo mínimo (por defecto un ejemplo en `raw.githubusercontent.com/elyra-ai/examples/...`). |
| `ELYRA_AUTO_CATALOG_NAME` | Nombre del metadato del catálogo creado (por defecto `rhods_tl_kfp_minimal_url`). |

## Fallo por incompatibilidad pipeline / runtime

Si ves un error de tipo *runtime configuration type does not match pipeline's runtime type*, el `retrain.pipeline` está fijado a `KUBEFLOW_PIPELINES`; el runtime elegido debe ser de ese tipo (crea otro runtime o cambia el pipeline en el editor visual).